# arXiv LLM Pipeline - Data Exploration

This notebook provides an interactive environment for exploring the arXiv data pipeline.

In [4]:
# Setup
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config.settings import settings

print(f"Environment: {settings.environment}")
print(f"MinIO Endpoint: {settings.minio.endpoint}")

Environment: Environment.DEVELOPMENT
MinIO Endpoint: localhost:9000


## 1. Data Collection

Collect sample papers from arXiv.

In [5]:
from src.collectors.arxiv_client import ArxivClient
import asyncio

async def collect_samples():
    client = ArxivClient()
    papers = await client.collect_papers(
        categories=["cs.LG"],
        limit=10,
        download_pdfs=False
    )
    return papers

papers = await collect_samples()
print(f"Collected {len(papers)} papers")

{"rate_limit": 3.0, "download_dir": "c:\\Users\\Thienhb\\Workspace\\Personal\\train-llm-with-arxiv-data\\notebooks\\..\\data\\pdfs", "event": "ArxivClient initialized", "level": "info", "timestamp": "2025-12-23T07:22:13.432959Z"}
{"query": "(cat:cs.LG)", "max_results": 10, "event": "Starting arXiv search", "level": "info", "timestamp": "2025-12-23T07:22:13.433968Z"}
Requesting page (first: True, try: 0): https://export.arxiv.org/api/query?search_query=%28cat%3Acs.LG%29&id_list=&sortBy=submittedDate&sortOrder=descending&start=0&max_results=100
Got first page: 100 of 246240 total results
{"total_papers": 10, "event": "Search complete", "level": "info", "timestamp": "2025-12-23T07:22:16.055717Z"}
{"total_papers": 10, "event": "Collection complete", "level": "info", "timestamp": "2025-12-23T07:22:16.056716Z"}
Collected 10 papers


In [6]:
# Display paper metadata
for i, paper in enumerate(papers[:3]):
    print(f"\n--- Paper {i+1} ---")
    print(f"Title: {paper.title}")
    print(f"Authors: {', '.join(paper.authors[:3])}...")
    print(f"Categories: {paper.categories}")
    print(f"Abstract: {paper.abstract[:200]}...")


--- Paper 1 ---
Title: Pushing the Frontier of Audiovisual Perception with Large-Scale Multimodal Correspondence Learning
Authors: Apoorv Vyas, Heng-Jui Chang, Cheng-Fu Yang...
Categories: ['cs.SD', 'cs.CV', 'cs.LG']
Abstract: We introduce Perception Encoder Audiovisual, PE-AV, a new family of encoders for audio and video understanding trained with scaled contrastive learning. Built on PE, PE-AV makes several key contributi...

--- Paper 2 ---
Title: Bottom-up Policy Optimization: Your Language Model Policy Secretly Contains Internal Policies
Authors: Yuqiao Tan, Minzheng Wang, Shizhu He...
Categories: ['cs.LG', 'cs.AI', 'cs.CL']
Abstract: Existing reinforcement learning (RL) approaches treat large language models (LLMs) as a single unified policy, overlooking their internal mechanisms. Understanding how policy evolves across layers and...

--- Paper 3 ---
Title: Deep Legendre Transform
Authors: Aleksey Minabutdinov, Patrick Cheridito...
Categories: ['cs.LG', 'math.OC']
Abstract: We i

## 2. Text Processing

Explore text normalization functions.

In [7]:
from src.processing.normalization import normalize_text

# Example text with LaTeX and HTML
sample_text = """
<p>We propose a new method for $\\alpha$-divergence minimization.</p>
The loss function is $L = \\frac{1}{n}\\sum_{i=1}^{n} \\ell(x_i)$.
Contact: researcher@university.edu
"""

normalized = normalize_text(sample_text)
print("Original:")
print(sample_text)
print("\nNormalized:")
print(normalized)

AttributeError: module 'socketserver' has no attribute 'UnixStreamServer'

In [ ]:
from src.processing.content_moderation import detect_pii, redact_pii

# PII detection
pii_types = detect_pii(sample_text)
print(f"PII types found: {pii_types}")

# Redaction
redacted = redact_pii(sample_text)
print(f"\nRedacted text:\n{redacted}")

## 3. Tokenization

Explore tokenization options.

In [ ]:
# This would require a trained tokenizer
# from src.tokenization.trainer import TokenizerTrainer

print("Tokenizer training requires text data.")
print("Run: arxiv-llm tokenize train --input <text_file>")

## 4. Model Architecture

Explore model configurations.

In [8]:
from src.training.model import TransformerConfig, TransformerLM

# Compare model sizes
configs = {
    "Small (~125M)": TransformerConfig.small(),
    "Medium (~350M)": TransformerConfig.medium(),
    "Large (~760M)": TransformerConfig.large(),
}

for name, config in configs.items():
    model = TransformerLM(config)
    params = model.num_parameters()
    print(f"{name}: {params:,} parameters")

Small (~125M): 111,204,864 parameters
Medium (~350M): 337,176,576 parameters
Large (~760M): 751,971,840 parameters


In [ ]:
# Small model test
import torch

config = TransformerConfig(
    vocab_size=1000,
    hidden_size=64,
    num_hidden_layers=2,
    num_attention_heads=4,
)

model = TransformerLM(config)
input_ids = torch.randint(0, 1000, (1, 20))

outputs = model(input_ids)
print(f"Input shape: {input_ids.shape}")
print(f"Output logits shape: {outputs['logits'].shape}")

## 5. Next Steps

1. Collect more papers: `arxiv-llm collect arxiv --limit 1000`
2. Process data: `arxiv-llm process pipeline -i raw -o processed`
3. Train tokenizer: `arxiv-llm tokenize train -i <text_file>`
4. Start training: `arxiv-llm train start -d <data_path>`